# Loading data 

In [1]:
import pyspark.pandas as ps

catalog = "workspace.default"

df_orders = ps.read_table(f"{catalog}.bronze_orders")
df_products = ps.read_table(f"{catalog}.bronze_products")
df_sellers = ps.read_table(f"{catalog}.bronze_sellers")
df_geolocation = ps.read_table(f"{catalog}.bronze_geolocation")
df_payments = ps.read_table(f"{catalog}.bronze_order_payments")
df_customers = ps.read_table(f"{catalog}.bronze_customers")
df_items = ps.read_table(f"{catalog}.bronze_order_items")
df_reviews = ps.read_table(f"{catalog}.bronze_order_reviews")
df_translation = ps.read_table(f"{catalog}.bronze_category_translation")

list_of_tables = [df_orders, df_products, df_sellers, df_geolocation, df_payments, df_customers, df_items, df_reviews, df_translation]




FileNotFoundError: [Errno 2] No such file or directory: 'workspace.default.bronze_orders'

# The database structure 
![image_1776258122474.png](./image_1776258122474.png "image_1776258122474.png")

In [0]:
display(df_orders.head(5))
df_orders.shape


/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


(99441, 8)

In [0]:
for table in list_of_tables:
    print(table.dtypes)
    print()

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

product_id                    object
product_category_name         object
product_name_lenght            int32
product_description_lenght     int32
product_photos_qty             int32
product_weight_g               int32
product_length_cm              int32
product_height_cm              int32
product_width_cm               int32
dtype: object

seller_id                 object
seller_zip_code_prefix     int32
seller_city               object
seller_state              object
dtype: object

geolocation_zip_code_prefix      int32
geolocation_lat                float64
geolocation_lng             

We can see that the data types was loaded correctly. But in reviews Dates are strings and review_score is also a string.

# Convert the wrong data types. 

In [0]:
display(df_reviews.head(5))

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


In [0]:
df_reviews['review_score']  = df_reviews['review_score'].astype(int)
df_reviews['review_creation_date'] = ps.to_datetime(df_reviews['review_creation_date'], format='%Y-%m-%d %H:%M:%S')
df_reviews['review_answer_timestamp'] = ps.to_datetime(df_reviews['review_answer_timestamp'], format='%Y-%m-%d %H:%M:%S')
display(df_reviews.head(5))

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z


# Translate products name to ang 

In [0]:
display(df_products.head(5))

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [0]:
df_products_eng = df_products.merge(df_translation, on='product_category_name', how='left')
df_products_eng = df_products_eng.drop(columns=['product_category_name'])

list_of_tables.append(df_products_eng)
list_of_tables.extend(df_products)

display(df_products_eng.head(5))

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
1e9e8ef04dbcff4541ed26657ea517e5,40,287,1,225,16,10,14,perfumery
3aa071139cb16b67ca9e5dea641aaa2f,44,276,1,1000,30,18,20,art
96bd76ec8810374ed1b65e291975717f,46,250,1,154,18,9,15,sports_leisure
cef67bcfe19066a932b7673e239eb23d,27,261,1,371,26,4,26,baby
9dc1a7de274444849c219cff195d0b71,37,402,4,625,20,17,13,housewares


# Merge products and product IDs.

In [0]:
df_items = df_items.merge(df_products_eng, how='left', on='product_id')
display(df_items.head(5))

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,58,598,4,650,28,9,14,cool_stuff
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,56,239,2,30000,50,30,40,pet_shop
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,59,695,2,3050,33,13,33,furniture_decor
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,42,480,1,200,16,10,15,perfumery
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,59,409,1,3750,35,40,30,garden_tools


# Use inputation on null values 

In [0]:
for table in list_of_tables:
    print(table)
    print(table.isnull().sum())

                             order_id                       customer_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0    e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13                    2017-10-18
1    53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45                    2018-08-13
2    47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23          2018-08-08 13:50:00           2018-08-17 18:06:29                    2018-09-04
3    949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82    delivered      2017-11-18 19:28:06 2017-11-18 19:

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1373: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
                           product_id                           product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
0    1e9e8ef04dbcff4541ed26657ea517e5                                      perfumaria                 40.0                       287.0                 1.0               225                 16                 10                14
1    3aa071139cb16b67ca9e5dea641aaa2f                                           artes                 44.0                       276.0                 1.0              1000                 30                 18               

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1373: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
                            seller_id  seller_zip_code_prefix                               seller_city seller_state
0    3442f8959a84dea7ee197c632cb2df15                   13023                                  campinas           SP
1    d1b65fc7debc3361ea86b5f14c68d2e2                   13844                                mogi guacu           SP
2    ce3ad9de960102d0677a81f5d0bb7b2d                   20031                            rio de janeiro           RJ
3    c0f3eea2e14555b6faeea3dd58c1b1c3                    4195                                 sao paulo           SP
4    51a04a8a6bdcb23deccc82b0b80742cf                   12914                         braganca p

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1373: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
     geolocation_zip_code_prefix  geolocation_lat  geolocation_lng geolocation_city geolocation_state
0                           1037       -23.545621       -46.639292        sao paulo                SP
1                           1046       -23.546081       -46.644820        sao paulo                SP
2                           1046       -23.546129       -46.642951        sao paulo                SP
3                           1041       -23.544392       -46.639499        sao paulo                SP
4                           1035       -23.541578       -46.641607        sao paulo                SP
5                           1012       -23.547762       -46.635361        são paulo                SP
6                           1047       -23.546273       -46.641225        sao paulo                SP
7                           1013       -23.546923       -46

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1373: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

# 
